# 12 - Reranker Ablation: A0 (baseline) / A1 (score-only) / A2 (pruned)

Three arms, per `guidance/retrieval_telemetry_and_reranking_design.md`'s ablation
table and `guidance/ANALYSIS_reranker_judgment_calls_2026-07-29.md`'s acceptance-bar
recommendation (Sec 2.6):

- **A0** = no reranking (same as notebook 11's `expanded` stage).
- **A1** = score-only (`rerank_top_n_blocks=None`, `min_score=0.0`) -- membership is
  provably unchanged from A0, so this isolates whether the cross-encoder's scores
  correlate with gold evidence at all, before any pruning-threshold decision.
- **A2** = pruned (`rerank_top_n_blocks=8`, the shipped default) -- the real
  cost/quality tradeoff.

Retrieval + expansion run **once per question** and are shared across all 3 arms --
they differ only in what happens after expansion, and retrieval dominates latency
(~30s/query) while reranking is <1s, so redoing retrieval 3x would be pure waste.

**Gate 0** (from the acceptance-bar doc): the block containing gold evidence should
rank top-3 by `relevanceScore` in >=24/31 questions on arm A1, or the reranker
should be considered not to have earned further investment.

In [1]:
import sys
import json
import time
from pathlib import Path

for p in [Path.cwd()] + list(Path.cwd().parents):
    if p.name == "ModelPipeline":
        MODEL_ROOT = p
        break
if str(MODEL_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT))

from finrag_ml_tg1.loaders.ml_config_loader import MLConfig
from finrag_ml_tg1.rag_modules_src.synthesis_pipeline.supply_lines import (
    init_rag_components, run_supply_line_2_rag,
)
from finrag_ml_tg1.rag_modules_src.rag_pipeline.reranker import CohereReranker
from finrag_ml_tg1.rag_modules_src.utilities.retrieval_telemetry import build_retrieval_telemetry
from finrag_ml_tg1.rag_modules_src.utilities.retrieval_metrics import score_query, aggregate, paired_delta

config = MLConfig()

rag = init_rag_components()          # reranker=None -- used for the shared retrieval+expansion pass
rag.retriever.enable_variants = False  # in-memory only, same as notebook 11

retrieval_cfg = dict(config.cfg["retrieval"])

reranker_a1 = CohereReranker(
    retrieval_config={**retrieval_cfg, "rerank_top_n_blocks": None, "rerank_min_score": 0.0},
    region=config.region, aws_access_key_id=config.aws_access_key,
    aws_secret_access_key=config.aws_secret_key,
)
reranker_a2 = CohereReranker(
    retrieval_config={**retrieval_cfg, "rerank_top_n_blocks": 8, "rerank_min_score": 0.0},
    region=config.region, aws_access_key_id=config.aws_access_key,
    aws_secret_access_key=config.aws_secret_key,
)

GOLD_PATH = MODEL_ROOT.parent / "MLFlow_POC" / "data" / "p3_gold_test_suite_31q.json"
gold = json.loads(GOLD_PATH.read_text())
print(f"Loaded {len(gold)} gold questions")

[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Local dev detected → LOCAL_CACHE mode
Loaded 31 gold questions


In [2]:
rows_a0, rows_a1, rows_a2 = [], [], []
gate0_hits = 0  # gold block ranks top-3 in A1
t_start = time.time()

for i, gq in enumerate(gold, start=1):
    try:
        # Single retrieval+expansion pass (rag.reranker is None here) -- unique_sents
        # is the actual pre-rerank SentenceRecord list, reused directly below for A1/A2
        # instead of re-running retrieval two more times.
        _, entities, bundle, unique_sents, _, base_telemetry = run_supply_line_2_rag(
            gq["question_text"], rag
        )
    except Exception as e:
        print(f"[{i:2d}/{len(gold)}] {gq['question_id']:12s} RETRIEVAL FAILED: {type(e).__name__}: {e}")
        continue

    query_text = gq["question_text"]
    a1_sents = reranker_a1.rerank(query_text, unique_sents)
    a2_sents = reranker_a2.rerank(query_text, unique_sents)

    tel_a1 = build_retrieval_telemetry(query_text, bundle, unique_sents, reranked_sents=a1_sents)
    tel_a2 = build_retrieval_telemetry(query_text, bundle, unique_sents, reranked_sents=a2_sents)

    row_a0 = score_query(base_telemetry, gq)
    row_a1 = score_query(tel_a1, gq)
    row_a2 = score_query(tel_a2, gq)
    rows_a0.append(row_a0)
    rows_a1.append(row_a1)
    rows_a2.append(row_a2)

    # Gate 0: does a gold sentence's containing block rank top-3 in A1's core_hits-equivalent?
    # Approximate via reranked_sentence_ids order (A1 keeps everything, reordered by score):
    # a "top-3 block" proxy = gold sentence id appears within the first 3 DISTINCT blocks'
    # worth of sentences at the front of the A1 reranked list.
    gold_ids = set(gq['evidence_sentence_ids'])
    a1_ids = tel_a1['reranked_sentence_ids'] or []
    # crude block-count proxy: count distinct leading run positions until we've seen 3 score
    # tiers is not directly available here; use rank-of-first-gold-hit <= 24 (~top 3 of ~8
    # avg blocks * ~3 sentences) as a practical proxy instead of re-deriving block boundaries.
    first_hit_rank = next((idx for idx, sid in enumerate(a1_ids, start=1) if sid in gold_ids), None)
    gate0_pass = first_hit_rank is not None and first_hit_rank <= 24
    gate0_hits += int(gate0_pass)

    print(f"[{i:2d}/{len(gold)}] {gq['question_id']:12s} "
          f"A0_mrr={row_a0['expanded_mrr']:.3f}  A1_mrr={row_a1['reranked_mrr']:.3f}  "
          f"A2_mrr={row_a2['reranked_mrr']:.3f}  gate0={'PASS' if gate0_pass else 'fail'}")

elapsed = time.time() - t_start
print(f"\n{len(rows_a0)}/{len(gold)} questions scored in {elapsed:.1f}s")
print(f"Gate 0 (gold block near top of A1 ranking): {gate0_hits}/{len(rows_a0)}")

[ 1/31] P3V2-Q001    A0_mrr=0.000  A1_mrr=0.000  A2_mrr=0.000  gate0=fail


[ 2/31] P3V2-Q002    A0_mrr=0.014  A1_mrr=0.020  A2_mrr=0.000  gate0=fail


[ 3/31] P3V2-Q003    A0_mrr=0.000  A1_mrr=0.000  A2_mrr=0.000  gate0=fail


[ 4/31] P3V2-Q004    A0_mrr=0.000  A1_mrr=0.000  A2_mrr=0.000  gate0=fail


[ 5/31] P3V2-Q005    A0_mrr=0.017  A1_mrr=0.200  A2_mrr=0.200  gate0=PASS


...

[27/31] P3V3-Q006    A0_mrr=0.091  A1_mrr=0.071  A2_mrr=0.071  gate0=PASS


[28/31] P3V3-Q007    A0_mrr=0.013  A1_mrr=0.024  A2_mrr=0.024  gate0=fail


[29/31] P3V3-Q008    A0_mrr=0.000  A1_mrr=0.000  A2_mrr=0.000  gate0=fail


[30/31] P3V3-Q009    A0_mrr=0.250  A1_mrr=0.250  A2_mrr=0.250  gate0=PASS


[31/31] P3V3-Q010    A0_mrr=0.018  A1_mrr=0.100  A2_mrr=0.100  gate0=PASS

31/31 questions scored in 91.4s
Gate 0 (gold block near top of A1 ranking): 15/31


In [3]:
print("=== A0 (baseline, no reranking) ===")
agg_a0 = aggregate(rows_a0)
for k in ("n", "expanded_recall@5", "expanded_recall@30", "expanded_mrr"):
    print(f"  {k:22s} {agg_a0['overall'].get(k)}")

print("\n=== A1 (score-only, membership unchanged) ===")
agg_a1 = aggregate(rows_a1)
for k in ("n", "reranked_recall@5", "reranked_recall@30", "reranked_mrr"):
    print(f"  {k:22s} {agg_a1['overall'].get(k)}")

print("\n=== A2 (pruned, top 8 blocks) ===")
agg_a2 = aggregate(rows_a2)
for k in ("n", "reranked_recall@5", "reranked_recall@30", "reranked_mrr"):
    print(f"  {k:22s} {agg_a2['overall'].get(k)}")

# Sanity check the design doc's own claim: A1 membership unchanged => recall@30 identical to A0
a0_r30 = agg_a0["overall"]["expanded_recall@30"]
a1_r30 = agg_a1["overall"]["reranked_recall@30"]
print(f"\nSanity check -- A0 vs A1 recall@30 should match (A1 doesn't prune): "
      f"{a0_r30:.4f} vs {a1_r30:.4f} {'MATCH' if abs(a0_r30 - a1_r30) < 1e-9 else 'MISMATCH -- investigate'}")

# Primary endpoint: gold-evidence survival under A2 pruning (rows where recall dropped to 0
# despite A0 having found something -- i.e. A2 pruning actively lost evidence A0 had)
a0_by_id = {r['question_id']: r for r in rows_a0}
a2_by_id = {r['question_id']: r for r in rows_a2}
survived = sum(
    1 for qid in a0_by_id
    if a0_by_id[qid]['expanded_recall@30'] == 0 or a2_by_id[qid]['reranked_recall@30'] >= a0_by_id[qid]['expanded_recall@30']
)
print(f"\nGold-evidence survival under A2 pruning: {survived}/{len(a0_by_id)} "
      f"(primary safety endpoint -- ship bar per ANALYSIS doc Sec 2.6 is >=30/31 overall)")

print("\n=== Paired A0 vs A2 deltas (per ANALYSIS doc Sec 2.1-2.2) ===")
for metric in ("expanded_recall@5", "expanded_mrr"):
    a2_metric = metric.replace("expanded_", "reranked_")
    rows_a0_renamed = [{**r, metric: r[metric]} for r in rows_a0]
    rows_a2_renamed = [{**r, metric: r.get(a2_metric)} for r in rows_a2]
    delta = paired_delta(rows_a0_renamed, rows_a2_renamed, metric, n_boot=10_000)
    print(f"  {metric}: mean_delta={delta['mean_delta']:.4f}  ci_95={delta['ci_95']}  "
          f"n_better={delta['n_better']} n_worse={delta['n_worse']} n_tied={delta['n_tied']}")

=== A0 (baseline, no reranking) ===
  n                      31
  expanded_recall@5      0.04301075268817204
  expanded_recall@30     0.059139784946236555
  expanded_mrr           0.02881149748302612

=== A1 (score-only, membership unchanged) ===
  n                      31
  reranked_recall@5      0.13709677419354838
  reranked_recall@30     0.47043010752688175
  reranked_mrr           0.07584929980274437

=== A2 (pruned, top 8 blocks) ===
  n                      31
  reranked_recall@5      0.13709677419354838
  reranked_recall@30     0.47043010752688175
  reranked_mrr           0.07351593813558185

Sanity check -- A0 vs A1 recall@30 should match (A1 doesn't prune): 0.0591 vs 0.4704 MISMATCH -- investigate

Gold-evidence survival under A2 pruning: 31/31 (primary safety endpoint -- ship bar per ANALYSIS doc Sec 2.6 is >=30/31 overall)

=== Paired A0 vs A2 deltas (per ANALYSIS doc Sec 2.1-2.2) ===
  expanded_recall@5: mean_delta=0.0941  ci_95=(0.0, 0.21505376344086022)  n_better=4 n_wo

## Interpretation (fill in after running)

- **Gate 0** result determines whether it's worth reading further: per the
  ANALYSIS doc, if the gold block doesn't rank near the top in the *unpruned*
  A1 arm on most questions, no pruning threshold in A2 will fix that -- the
  cross-encoder simply isn't discriminating well on this corpus.
- The **A0 vs A1 recall@30 sanity check** should show an exact match (A1 never
  drops a block, only reorders/scores) -- if it doesn't, that's a bug in the
  reranker's `_select()`/`min_score` logic, not a real finding, and needs fixing
  before trusting anything else here.
- **Gold-evidence survival** under A2 is the primary ship/no-ship endpoint --
  read it together with the acknowledged statistical limit from the ANALYSIS
  doc Sec 2.3: even a perfect 31/31 only bounds the true regression rate at
  ~9.2% (one-sided 95%), it does not prove safety.
- Per the ANALYSIS doc's power calculation (Sec 2.2), the paired-delta MRR
  needs to move roughly +0.07 to +0.13 before it's distinguishable from noise
  at n=31 -- read any smaller delta as "not yet resolved," not as "no effect."
